<a href="https://colab.research.google.com/github/EliasNoorzad/spacecraft-pose-estimation/blob/main/notebooks/yolo_test_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DLVS3-HST — YOLOv8s-Pose test evaluation

Evaluate the saved model on the held-out test set, then measure keypoint errors in pixels. The outputs below are from the completed smoke test; no training or PnP is included.

In [ ]:
!pip -q install ultralytics pyyaml

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml

# Find the test set and its labels.
candidates = [
    Path("/content/hst_yolo_pose"),
    Path("/content/drive/MyDrive/hst_yolo_pose_fixed"),
    Path("/content/drive/MyDrive/hst_yolo_pose"),
]

dataset_root = next(
    (p for p in candidates
     if (p / "images/test").is_dir()
     and (p / "labels/test").is_dir()),
    None
)

assert dataset_root is not None, "Test dataset not found"

# Make the saved dataset paths work in Colab.
config = {
    "path": str(dataset_root),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": {0: "HST"},
    "kpt_shape": [37, 3],
    "kpt_oks_sigmas": [0.10] * 37
}

yaml_path = "/content/hst_test.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(config, f)

# Use the checkpoint from training.
model = YOLO(
    "/content/drive/MyDrive/HST_YOLO_Runs/"
    "hst_pose_sigma_010/weights/best.pt"
)

# Report test-set metrics using YOLO.
metrics = model.val(
    data=yaml_path,
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    verbose=True,
    plots=True,
    project="/content/drive/MyDrive/HST_YOLO_Runs",
    name="final_test_evaluation",
    exist_ok=True
)

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-pose summary (fused): 81 layers, 12,391,704 parameters, 0 gradients, 33.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4734.8±1127.8 MB/s, size: 692.2 KB)
val: Scanning /content/hst_yolo_pose/labels/test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 21.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 4.4it/s 3.0s
                   all        100        100          1      0.999      0.995      0.882      0.876       0.86      0.789      0.542
Speed: 2.3ms preprocess, 13.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /content/drive/MyDrive/HST_YOLO_Runs/final_test_evaluation


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display

# Locate the test data

root = Path("/content/drive/MyDrive/hst_yolo_pose")

test_dir = Path("/content/hst_yolo_pose/images/test")

if not test_dir.exists():
    test_dir = root / "images/test"

model_path = (
    "/content/drive/MyDrive/HST_YOLO_Runs/"
    "hst_pose_sigma_010/weights/best.pt"
)

assert test_dir.exists(), "Test images not found"
assert (root / "0000000_metadata.json").exists(), "Metadata not found"

# Load annotated keypoints

with open(root / "0000000_metadata.json") as f:
    metadata = json.load(f)

def extract_frames(obj):
    if isinstance(obj, dict):
        if "ImagePath" in obj and "TargetKeypoints" in obj:
            yield obj
        else:
            for value in obj.values():
                yield from extract_frames(value)
    elif isinstance(obj, list):
        for item in obj:
            yield from extract_frames(item)

gt_by_name = {
    Path(frame["ImagePath"]).stem: frame
    for frame in extract_frames(metadata)
}

# Run the saved YOLO checkpoint

model = YOLO(model_path)

predictions = model.predict(
    source=str(test_dir),
    imgsz=640,
    conf=0.25,
    device=0,
    save=False,
    verbose=False
)

# Measure keypoint errors

records = []
missing_detections = 0

for result in predictions:

    image_name = Path(result.path).name
    stem = Path(result.path).stem

    if stem not in gt_by_name:
        continue

    if result.keypoints is None or len(result.keypoints.xy) == 0:
        missing_detections += 1
        continue

    # Predicted image coordinates.
    pred = result.keypoints.xy[0].cpu().numpy()

    # Get the annotated 2D keypoints.
    gt = np.array([
        [p["X"], p["Y"]]
        for p in gt_by_name[stem]["TargetKeypoints"]
    ], dtype=np.float64)

    assert pred.shape == gt.shape == (37, 2)

    h, w = result.orig_shape

    # Ignore ground-truth points outside the frame.
    valid = (
        np.isfinite(gt).all(axis=1)
        & (gt[:, 0] >= 0)
        & (gt[:, 0] < w)
        & (gt[:, 1] >= 0)
        & (gt[:, 1] < h)
    )

    for i in np.where(valid)[0]:

        error = np.linalg.norm(pred[i] - gt[i])

        records.append({
            "Image": image_name,
            "Keypoint": i + 1,
            "Pixel Error": error,
            "Zero Prediction": bool(np.all(pred[i] == 0))
        })

# Summarize pixel errors

df = pd.DataFrame(records)

assert not df.empty, "No matching predictions and annotations"

errors = df["Pixel Error"]

print("\n========== YOLO KEYPOINT ACCURACY ==========")

print(f"Test images processed:   {len(predictions)}")
print(f"Images evaluated:        {df['Image'].nunique()}")
print(f"Missing detections:      {missing_detections}")
print(f"Keypoints evaluated:     {len(df)}")
print(f"Zero predictions:        {df['Zero Prediction'].sum()}")

print(f"\nMean pixel error:        {errors.mean():.3f} px")
print(f"Median pixel error:      {errors.median():.3f} px")
print(f"90th percentile error:   {errors.quantile(0.90):.3f} px")
print(f"Maximum pixel error:     {errors.max():.3f} px")

print(f"\nWithin 5 pixels:         {(errors <= 5).mean()*100:.2f}%")
print(f"Within 10 pixels:        {(errors <= 10).mean()*100:.2f}%")
print(f"Within 20 pixels:        {(errors <= 20).mean()*100:.2f}%")

print("\n========== ERROR BY KEYPOINT ==========")

per_keypoint = df.groupby("Keypoint")["Pixel Error"].agg(
    ["mean", "median", "max"]
)

display(per_keypoint.round(3))

print("\n========== 10 WORST IMAGES ==========")

per_image = df.groupby("Image")["Pixel Error"].agg(
    ["mean", "median", "max"]
)

display(
    per_image.sort_values("mean", ascending=False)
    .head(10)
    .round(3)
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

========== YOLO KEYPOINT ACCURACY ==========
Test images processed:   100
Images evaluated:        99
Missing detections:      1
Keypoints evaluated:     3662
Zero predictions:        0

Mean pixel error:        34.887 px
Median pixel error:      19.527 px
90th percentile error:   79.822 px
Maximum pixel error:     412.991 px

Within 5 pixels:         8.41%
Within 10 pixels:        24.22%
Within 20 pixels:        50.85%

========== ERROR BY KEYPOINT ==========


,mean,median,max
Keypoint,,,
1,36.272,21.840,187.131
2,25.849,17.076,207.966
3,27.497,18.388,137.543
4,27.305,17.617,240.449
5,27.703,15.872,173.930
6,20.186,11.924,104.160
7,21.856,13.813,109.118
8,59.898,30.420,337.920
9,65.839,40.541,345.130



========== 10 WORST IMAGES ==========


,mean,median,max
Image,,,
output_0000095.png,191.324,172.798,412.991
output_0000777.png,169.501,130.000,364.056
output_0000387.png,121.633,100.410,296.872
output_0000906.png,112.694,98.007,282.099
output_0000233.png,103.422,84.301,222.733
output_0000887.png,101.298,95.488,245.821
output_0000370.png,93.805,74.405,218.663
output_0000875.png,93.558,76.645,279.155
output_0000747.png,90.057,72.910,203.316
